Notebook purpose (how this analysis supports the thesis)
This notebook operationalises the thesis proposal “Governance‑Ready Fraud Decisioning for Transaction Monitoring: Evidence‑Grounded Narrative Explanations with Tiered Guardrails” (Andrew Murtagh, 22 Feb 2026) by analysing a minimal, reproducible run of the pipeline:

a leakage‑controlled baseline fraud model on IEEE‑CIS (temporal quantile split),
per‑event Evidence Objects (EOs) as the closed‑world “contract” for explanations, and
persona‑specific narratives generated under two explanation conditions (templates vs constrained LLM), validated by a tiered validator with retries and template fallback.
The outputs here are intended for thesis figures/tables and for diagnosing governance‑relevant failure modes (schema violations, closed‑world breaches, missing disclosures, and action inconsistency).

What is being tested (research questions mapped to notebook outputs)
This analysis focuses on the proposal’s core evaluation axes:

Faithfulness (RQ1): Do narratives preserve EO driver membership/order/direction and remain consistent with the EO’s recommended action class?
Stability (RQ2): Where available, do narratives behave predictably under perturbations or controlled stress (e.g., thin‑file / low‑coverage conditions)?
Note: The notebook now includes a compact perturbation experiment, but full perturbation/randomisation sanity checks remain outside the scope of this run.
Decision utility (RQ3, proxy): Do persona narratives provide a usable summary (risk, drivers, disclosures, action) without introducing unsupported facts?
Governance contract: Evidence Objects (EOs) and closed‑world grounding
The EO is treated as the single source of truth for downstream narratives. Narratives must remain “closed‑world” grounded: they may reference only fields present in the EO (score/band/action, top‑K drivers and directions, monitoring/drift flags, and evidence‑strength/coverage signals). Any narrative content that cannot be traced to EO fields is considered a governance failure.

Compared narrative conditions
This notebook compares (or prepares comparison for) two explanation conditions defined in the proposal:

Template narratives (deterministic baseline): guaranteed schema/grounding by construction.
Constrained LLM narratives (structured outputs): LLM returns JSON constrained to a strict schema; outputs are accepted only if they pass a defence‑in‑depth validator. Failures are retried; persistent failures trigger template fallback. Retry counts and fallback reasons are treated as empirical results, not just engineering details.
Thin‑file and monitoring disclosures (required governance behaviour)
A central requirement is correct uncertainty communication when evidence is limited. For EOs flagged as thin‑file / low evidence_strength / low coverage, narratives must include an evidence limitation disclosure (and, where applicable, drift monitoring disclosures). The notebook explicitly tracks whether disclosures appear when required, and how often validation fails due to missing disclosures.

Reproducibility and scope of this notebook
This notebook is run against the following reproducible artifacts (EO JSONL, narrative JSONL, split manifest, and metrics files) produced by the pipeline scripts. The intent is not to claim production‑grade fraud performance, but to evaluate whether the EO → narrative → validator protocol can produce governance‑ready explanations with measurable faithfulness and robust handling of thin‑file cases.

# Analysis roadmap

This notebook evaluates a reproducible run of the EO → narrative → validation pipeline for governance-ready fraud explanation analysis. The notebook is organised into the following sections.

## 1. Setup and reproducibility
**Cell 0 — Configuration and data loading**

Loads reproducible artifacts, resolves repository-relative paths, applies fail-fast checks for required files, and constructs the core analysis tables used throughout the notebook.

## 2. Input artifact inspection
**Cell 1 — EO quick inspection**  
**Cell 2 — LLM narrative quick inspection**  
**Cell 3 — EO × LLM merged inspection**

Performs lightweight sanity checks on the loaded Evidence Objects (EOs), generated LLM narratives, and the merged EO–LLM analysis set. These cells confirm that the expected fields are present and that event identifiers align correctly.

## 3. Canonical analysis frames
**Cell 4 — Canonical analysis frames + guardrails**

Constructs canonical analysis tables for LLM outputs, template outputs, and cross-condition comparison. Narrative fields are flattened into analysis-friendly columns while preserving the original structured records.

## 4. Faithfulness evaluation
**Cell 5 — Faithfulness metrics**

Computes first-order faithfulness indicators from EO-grounded narratives:
- driver overlap with EO top-K drivers,
- direction consistency between EO drivers and narrative statements,
- action consistency between EO recommendation and narrative recommendation.

These metrics operationalise the notebook’s main RQ1 checks.

## 5. Summary tables for thesis reporting
**Cell 6 — compact summary table for thesis text/tables**

Produces a compact headline summary of the main LLM-condition results for direct reuse in thesis tables or results text.

## 6. Stratified governance analysis
**Cell 7 — metrics by evidence strength**  
**Cell 8 — metrics by thin-file flag**  
**Cell 9 — check whether required evidence disclosures appear for thin-file cases**

Examines whether explanation behaviour differs under lower-evidence conditions, especially for thin-file cases. These cells focus on governance-relevant uncertainty communication and disclosure compliance.

## 7. Embedded-vs-recomputed metric checks
**Cell 10 — compare notebook-computed metrics with embedded metrics from LLM narrative JSONL**

Compares notebook-recomputed metrics against metrics embedded in the narrative artifacts. This serves as a reproducibility and consistency check on pipeline outputs.

## 8. Cross-condition normalisation for comparison
**Cell 11 — normalise LLM and template outputs into a common comparison schema**

Normalises LLM and template outputs into a single common schema to support condition comparison, downstream aggregation, and comparative tables/figures.

## Interpretation note
This notebook is intended to assess governance-readiness of explanation outputs rather than to claim production-grade fraud detection performance. Reported results should therefore be interpreted as evidence about the EO-grounded explanation protocol, disclosure behaviour, and validation robustness under the current reproducible run.

In [1]:
# Cell 0 — Configuration and data loading

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

# Optional plotting dependency
try:
    import matplotlib.pyplot as plt  # noqa: F401
    HAS_MPL = True
except Exception:
    plt = None  # type: ignore
    HAS_MPL = False


def _find_repo_root(start: Optional[Path] = None) -> Path:
    """Walk upward until a repository marker is found."""
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    return cur


def read_jsonl(path: Path, *, limit: int | None = None) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON at {path} line {i}: {e}") from e
            if limit is not None and len(rows) >= limit:
                break
    return rows


def _assert_columns(df: pd.DataFrame, cols: List[str], *, name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise AssertionError(
            f"{name} missing required columns: {missing}\n"
            f"Found: {list(df.columns)}"
        )


REPO_ROOT = _find_repo_root()
ARTIFACTS = REPO_ROOT / "artifacts" / "baselines" / "lgbm_numeric_v1_subsample"
OUTPUT = REPO_ROOT / "output"

ARTIFACTS = Path(os.getenv("THESIS_ARTIFACTS_DIR", str(ARTIFACTS))).expanduser().resolve()
OUTPUT = Path(os.getenv("THESIS_OUTPUT_DIR", str(OUTPUT))).expanduser().resolve()

EO_PATH = ARTIFACTS / "eos_test_with_drivers.jsonl"
OPS_TRIAGE_LLM_PATH = OUTPUT / "narratives_ops_triage_llm.jsonl"
OPS_TRIAGE_TEMPLATE_PATH = OUTPUT / "narratives_ops_triage_template.jsonl"

METRICS_PATH = ARTIFACTS / "metrics.json"
THIN_THICK_PATH = ARTIFACTS / "thin_thick_metrics.json"

required = {
    "EO_PATH": EO_PATH,
    "OPS_TRIAGE_LLM_PATH": OPS_TRIAGE_LLM_PATH,
    "OPS_TRIAGE_TEMPLATE_PATH": OPS_TRIAGE_TEMPLATE_PATH,
}
missing = {k: str(p) for k, p in required.items() if not p.exists()}
if missing:
    msg = "\n".join([f"- {k}: {v}" for k, v in missing.items()])
    raise FileNotFoundError(
        "Required artifact(s) not found:\n"
        f"{msg}\n\n"
        "Set THESIS_ARTIFACTS_DIR / THESIS_OUTPUT_DIR or regenerate the pipeline outputs."
    )

eo_rows = read_jsonl(EO_PATH)
eo_df = pd.DataFrame(eo_rows)
_assert_columns(
    eo_df,
    ["event_id", "top_drivers", "recommended_action_class", "thin_file_flag", "evidence_strength"],
    name="eo_df",
)
eo_df["event_id"] = eo_df["event_id"].astype(str)

llm_rows = read_jsonl(OPS_TRIAGE_LLM_PATH)
llm_df = pd.DataFrame(llm_rows)
_assert_columns(llm_df, ["eo_event_id", "persona", "engine", "narrative", "metrics"], name="llm_df")
llm_df["eo_event_id"] = llm_df["eo_event_id"].astype(str)

tmpl_rows = read_jsonl(OPS_TRIAGE_TEMPLATE_PATH)
tmpl_df = pd.DataFrame(tmpl_rows) if tmpl_rows else pd.DataFrame()

eo_llm_df = eo_df.merge(
    llm_df,
    left_on="event_id",
    right_on="eo_event_id",
    how="inner",
    suffixes=("_eo", "_llm"),
)
if eo_llm_df.empty:
    raise AssertionError(
        "Merged EO × LLM is empty (likely event_id mismatch).\n"
        f"Example EO event_id: {eo_df['event_id'].iloc[0] if len(eo_df) else 'N/A'}\n"
        f"Example LLM eo_event_id: {llm_df['eo_event_id'].iloc[0] if len(llm_df) else 'N/A'}"
    )

print("REPO_ROOT:", REPO_ROOT)
print("ARTIFACTS:", ARTIFACTS)
print("OUTPUT:", OUTPUT)
print("HAS_MPL:", HAS_MPL, "(matplotlib optional)")
print(f"EO rows: {len(eo_df)} | LLM rows: {len(llm_df)} | eo_llm rows: {len(eo_llm_df)}")
print("EO thin_file_flag rate:", float(eo_df["thin_file_flag"].mean()))
print(
    "Optional metrics present:",
    "metrics.json" if METRICS_PATH.exists() else "metrics.json (missing)",
    "|",
    "thin_thick_metrics.json" if THIN_THICK_PATH.exists() else "thin_thick_metrics.json (missing)",
)

assert len(eo_df) > 0
assert len(llm_df) > 0
assert len(eo_llm_df) > 0

REPO_ROOT: /home/amurtagh/repos/miniature-fishstick
ARTIFACTS: /home/amurtagh/repos/miniature-fishstick/artifacts/baselines/lgbm_numeric_v1_subsample
OUTPUT: /home/amurtagh/repos/miniature-fishstick/output
HAS_MPL: True (matplotlib optional)
EO rows: 20000 | LLM rows: 200 | eo_llm rows: 200
EO thin_file_flag rate: 0.7909
Optional metrics present: metrics.json | thin_thick_metrics.json


In [ ]:
# Cell 0a — Ensure OPENAI_API_KEY is set for this kernel session\n\nimport getpass\n\nif not os.getenv("OPENAI_API_KEY"):\n    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY (input hidden): ")\n\nprint("OPENAI_API_KEY set for this kernel session:", bool(os.getenv("OPENAI_API_KEY")))\n

In [ ]:
# Cell 0b — Run unconstrained ablation generation via subprocess\n\nimport shlex\nimport subprocess\n\nUNCONSTRAINED_ABLATION_DIR = OUTPUT / "unconstrained_ablation"\nUNCONSTRAINED_ABLATION_DIR.mkdir(parents=True, exist_ok=True)\n\nmodel_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")\ncmd = shlex.split(\n    f"python scripts/generate_openai_narratives_v1.py --generation-mode unconstrained_json_attempt --eos-jsonl artifacts/baselines/lgbm_numeric_v1_subsample/eos_test_with_drivers.jsonl --persona ops_triage --k 5 --max-n 200 --max-retries 3 --model {model_name}"\n)\n\nsubprocess.run(cmd, check=True, cwd=REPO_ROOT, env=os.environ.copy())\nprint("Unconstrained ablation output directory:", UNCONSTRAINED_ABLATION_DIR)\n

In [2]:
# Cell 1 — EO quick inspection
print("Number of EO records:", len(eo_df))
eo_df[[
    "event_id",
    "score",
    "risk_band",
    "recommended_action_class",
    "thin_file_flag",
    "evidence_strength"
]].head(5)

Number of EO records: 20000


,event_id,score,risk_band,recommended_action_class,thin_file_flag,evidence_strength
0,3488961,0.005848,LOW,allow,True,LOW
1,3488968,0.000505,LOW,allow,True,LOW
2,3488985,0.004906,LOW,allow,True,LOW
3,3489002,0.002055,LOW,allow,True,LOW
4,3489003,0.016481,LOW,allow,True,LOW


In [3]:
# Cell 2 — LLM narrative quick inspection
print("Number of LLM narrative records:", len(llm_df))
llm_df[["eo_event_id", "persona", "engine"]].head(5)

Number of LLM narrative records: 200


,eo_event_id,persona,engine
0,3488961,ops_triage,llm
1,3488968,ops_triage,llm
2,3488985,ops_triage,llm
3,3489002,ops_triage,llm
4,3489003,ops_triage,llm


In [4]:
# Cell 3 — EO × LLM merged inspection

print("Number of EO × LLM merged records:", len(eo_llm_df))
eo_llm_df[
    [
        "event_id",
        "score",
        "risk_band",
        "recommended_action_class",
        "thin_file_flag",
        "evidence_strength",
    ]
].head(5)

Number of EO × LLM merged records: 200


,event_id,score,risk_band,recommended_action_class,thin_file_flag,evidence_strength
0,3488961,0.005848,LOW,allow,True,LOW
1,3488968,0.000505,LOW,allow,True,LOW
2,3488985,0.004906,LOW,allow,True,LOW
3,3489002,0.002055,LOW,allow,True,LOW
4,3489003,0.016481,LOW,allow,True,LOW


In [5]:
# Cell 4 — Canonical analysis frames + guardrails


# --- helper: flatten selected narrative fields safely ---
def _extract_narrative_fields(df: pd.DataFrame, narrative_col: str = "narrative") -> pd.DataFrame:
    out = df.copy()

    def _get(d, key, default=None):
        return d.get(key, default) if isinstance(d, dict) else default

    out["narrative_summary"] = out[narrative_col].apply(lambda d: _get(d, "summary"))
    out["narrative_action_recommendation"] = out[narrative_col].apply(lambda d: _get(d, "action_recommendation"))
    out["narrative_drivers_used"] = out[narrative_col].apply(lambda d: _get(d, "drivers_used", []))
    out["narrative_driver_statements"] = out[narrative_col].apply(lambda d: _get(d, "driver_statements", []))
    out["narrative_disclosures"] = out[narrative_col].apply(lambda d: _get(d, "disclosures", []))
    out["narrative_monitoring_flags"] = out[narrative_col].apply(lambda d: _get(d, "monitoring_flags", []))
    return out

# --- canonical LLM frame ---
llm_analysis_df = _extract_narrative_fields(eo_llm_df)

# --- canonical template frame ---
if len(tmpl_df) > 0:
    tmpl_df = tmpl_df.copy()
    if "eo_event_id" in tmpl_df.columns:
        tmpl_df["eo_event_id"] = tmpl_df["eo_event_id"].astype(str)

    tmpl_eo_df = eo_df.merge(
        tmpl_df,
        left_on="event_id",
        right_on="eo_event_id",
        how="inner",
        suffixes=("_eo", "_tmpl"),
    )
    tmpl_analysis_df = _extract_narrative_fields(tmpl_eo_df)
else:
    tmpl_eo_df = pd.DataFrame()
    tmpl_analysis_df = pd.DataFrame()

# --- comparison frame: LLM joined to template on event_id ---
if not tmpl_analysis_df.empty:
    llm_cols = [
        "event_id",
        "score",
        "risk_band",
        "recommended_action_class",
        "thin_file_flag",
        "evidence_strength",
        "top_drivers",
        "narrative",
        "metrics",
        "narrative_summary",
        "narrative_action_recommendation",
        "narrative_drivers_used",
        "narrative_driver_statements",
        "narrative_disclosures",
        "narrative_monitoring_flags",
        "overlap_at_5",
        "direction_accuracy",
        "action_consistency",
    ]
    llm_cols = [c for c in llm_cols if c in llm_analysis_df.columns]

    tmpl_cols = [
        "event_id",
        "narrative",
        "metrics",
        "narrative_summary",
        "narrative_action_recommendation",
        "narrative_drivers_used",
        "narrative_driver_statements",
        "narrative_disclosures",
        "narrative_monitoring_flags",
    ]
    tmpl_cols = [c for c in tmpl_cols if c in tmpl_analysis_df.columns]

    compare_df = llm_analysis_df[llm_cols].merge(
        tmpl_analysis_df[tmpl_cols],
        on="event_id",
        how="inner",
        suffixes=("_llm", "_tmpl"),
    )
else:
    compare_df = pd.DataFrame()

print("llm_analysis_df:", llm_analysis_df.shape)
print("tmpl_analysis_df:", tmpl_analysis_df.shape)
print("compare_df:", compare_df.shape)

llm_analysis_df: (200, 28)
tmpl_analysis_df: (200, 28)
compare_df: (200, 23)


In [6]:
# Cell 5 — Faithfulness metrics

def overlap_at_k(row, k=5):
    top_drivers = row.get("top_drivers", [])
    narrative = row.get("narrative", {})
    if not isinstance(top_drivers, list) or not isinstance(narrative, dict):
        return None
    eo_drivers = [d.get("name") for d in top_drivers[:k] if isinstance(d, dict)]
    llm_drivers = narrative.get("drivers_used", [])
    if not eo_drivers or not isinstance(llm_drivers, list):
        return None
    return len(set(eo_drivers) & set(llm_drivers)) / float(k)


def direction_accuracy(row, k=5):
    top_drivers = row.get("top_drivers", [])
    narrative = row.get("narrative", {})
    if not isinstance(top_drivers, list) or not isinstance(narrative, dict):
        return None
    eo_drv = {
        d.get("name"): d.get("direction")
        for d in top_drivers[:k]
        if isinstance(d, dict) and d.get("name") is not None
    }
    llm_statements = narrative.get("driver_statements", [])
    if not isinstance(llm_statements, list):
        return None
    llm_dir = {
        d.get("name"): d.get("direction")
        for d in llm_statements
        if isinstance(d, dict) and d.get("name") is not None
    }
    overlap_names = set(eo_drv.keys()) & set(llm_dir.keys())
    if not overlap_names:
        return None
    correct = sum(eo_drv[name] == llm_dir[name] for name in overlap_names)
    return correct / len(overlap_names)


def action_consistency(row):
    narrative = row.get("narrative", {})
    if not isinstance(narrative, dict):
        return None
    eo_action = row.get("recommended_action_class")
    llm_action = narrative.get("action_recommendation")
    return eo_action == llm_action


eo_llm_df["overlap_at_5"] = eo_llm_df.apply(overlap_at_k, axis=1)
eo_llm_df["direction_accuracy"] = eo_llm_df.apply(direction_accuracy, axis=1)
eo_llm_df["action_consistency"] = eo_llm_df.apply(action_consistency, axis=1)

print("Mean overlap@5:", eo_llm_df["overlap_at_5"].mean())
print("Mean direction accuracy:", eo_llm_df["direction_accuracy"].mean())
print("Action consistency rate:", eo_llm_df["action_consistency"].mean())

Mean overlap@5: 1.0
Mean direction accuracy: 1.0
Action consistency rate: 1.0


In [7]:
# Cell 6 — compact summary table for thesis text/tables

llm_summary_df = pd.DataFrame({
    "metric": [
        "n_eo_llm",
        "mean_overlap_at_5",
        "mean_direction_accuracy",
        "action_consistency_rate",
        "thin_file_rate"
    ],
    "value": [
        len(eo_llm_df),
        eo_llm_df["overlap_at_5"].mean(),
        eo_llm_df["direction_accuracy"].mean(),
        eo_llm_df["action_consistency"].mean(),
        eo_llm_df["thin_file_flag"].mean()
    ]
})

llm_summary_df

,metric,value
0,n_eo_llm,200.0
1,mean_overlap_at_5,1.0
2,mean_direction_accuracy,1.0
3,action_consistency_rate,1.0
4,thin_file_rate,1.0


In [8]:
# Cell 7 — metrics by evidence strength

llm_by_evidence_df = eo_llm_df.groupby("evidence_strength").agg(
    n=("event_id", "count"),
    overlap_at_5_mean=("overlap_at_5", "mean"),
    direction_accuracy_mean=("direction_accuracy", "mean"),
    action_consistency_rate=("action_consistency", "mean"),
)

llm_by_evidence_df

,n,overlap_at_5_mean,direction_accuracy_mean,action_consistency_rate
evidence_strength,,,,
LOW,200,1.0,1.0,1.0


In [9]:
# Cell 8 — metrics by thin-file flag

llm_by_thin_file_df = eo_llm_df.groupby("thin_file_flag").agg(
    n=("event_id", "count"),
    overlap_at_5_mean=("overlap_at_5", "mean"),
    direction_accuracy_mean=("direction_accuracy", "mean"),
    action_consistency_rate=("action_consistency", "mean"),
)

llm_by_thin_file_df

,n,overlap_at_5_mean,direction_accuracy_mean,action_consistency_rate
thin_file_flag,,,,
True,200,1.0,1.0,1.0


In [10]:
# Cell 9 — check whether required evidence disclosures appear for thin-file cases

def has_evidence_disclosure(narr):
    if not isinstance(narr, dict):
        return False
    disclosures = narr.get("disclosures", {})
    if not isinstance(disclosures, dict):
        return False
    val = disclosures.get("evidence")
    return isinstance(val, str) and len(val.strip()) > 0

eo_llm_df["has_evidence_disclosure"] = eo_llm_df["narrative"].apply(has_evidence_disclosure)

llm_disclosure_summary_df = eo_llm_df.groupby("thin_file_flag").agg(
    n=("event_id", "count"),
    disclosure_rate=("has_evidence_disclosure", "mean"),
)

llm_disclosure_summary_df

,n,disclosure_rate
thin_file_flag,,
True,200,1.0


In [11]:
# Cell 10 — compare notebook-computed metrics with embedded metrics from LLM narrative JSONL

print("Template rows:", len(tmpl_df))
print("Template columns:", list(tmpl_df.columns))

def _metric_from_metrics_col(df: pd.DataFrame, key: str):
    if "metrics" not in df.columns:
        return pd.Series(dtype=float)
    vals = df["metrics"].apply(
        lambda d: d.get(key) if isinstance(d, dict) else None
    )
    return pd.to_numeric(vals, errors="coerce")

llm_embedded_overlap = _metric_from_metrics_col(llm_df, "overlap_at_K")
llm_embedded_direction = _metric_from_metrics_col(llm_df, "direction_accuracy")
llm_embedded_action = _metric_from_metrics_col(llm_df, "action_consistency")

metric_compare_df = pd.DataFrame([{
    "notebook_overlap_mean": eo_llm_df["overlap_at_5"].mean(),
    "embedded_overlap_mean": llm_embedded_overlap.mean(),
    "notebook_direction_mean": eo_llm_df["direction_accuracy"].mean(),
    "embedded_direction_mean": llm_embedded_direction.mean(),
    "notebook_action_mean": eo_llm_df["action_consistency"].mean(),
    "embedded_action_mean": llm_embedded_action.mean(),
}])

metric_compare_df

Template rows: 200
Template columns: ['eo_event_id', 'persona', 'engine', 'run_metadata', 'narrative', 'metrics']


,notebook_overlap_mean,embedded_overlap_mean,notebook_direction_mean,embedded_direction_mean,notebook_action_mean,embedded_action_mean
0,1.0,1.0,1.0,1.0,1.0,1.0


In [12]:
# Cell 11 — normalise LLM and template outputs into a common comparison schema

def extract_llm_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["event_id_norm"] = out["eo_event_id"].astype(str)
    out["condition"] = "llm"

    out["summary_text"] = out["narrative"].apply(
        lambda x: x.get("summary") if isinstance(x, dict) else None
    )
    out["drivers_used_norm"] = out["narrative"].apply(
        lambda x: x.get("drivers_used", []) if isinstance(x, dict) else []
    )
    out["driver_statements_norm"] = out["narrative"].apply(
        lambda x: x.get("driver_statements", []) if isinstance(x, dict) else []
    )
    out["action_recommendation_norm"] = out["narrative"].apply(
        lambda x: x.get("action_recommendation") if isinstance(x, dict) else None
    )
    out["disclosures_norm"] = out["narrative"].apply(
        lambda x: x.get("disclosures", {}) if isinstance(x, dict) else {}
    )

    keep = [
        "event_id_norm",
        "condition",
        "persona",
        "engine",
        "summary_text",
        "drivers_used_norm",
        "driver_statements_norm",
        "action_recommendation_norm",
        "disclosures_norm",
        "metrics",
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep]


def extract_template_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "event_id" in out.columns:
        out["event_id_norm"] = out["event_id"].astype(str)
    elif "eo_event_id" in out.columns:
        out["event_id_norm"] = out["eo_event_id"].astype(str)
    else:
        raise AssertionError("Template dataframe needs either 'event_id' or 'eo_event_id'")

    out["condition"] = "template"
    if "persona" not in out.columns:
        out["persona"] = None
    if "engine" not in out.columns:
        out["engine"] = "template"

    out["summary_text"] = out["narrative"].apply(
        lambda x: x.get("summary") if isinstance(x, dict) else None
    )
    out["drivers_used_norm"] = out["narrative"].apply(
        lambda x: x.get("drivers_used", []) if isinstance(x, dict) else []
    )
    out["driver_statements_norm"] = out["narrative"].apply(
        lambda x: x.get("driver_statements", []) if isinstance(x, dict) else []
    )
    out["action_recommendation_norm"] = out["narrative"].apply(
        lambda x: x.get("action_recommendation") if isinstance(x, dict) else None
    )
    out["disclosures_norm"] = out["narrative"].apply(
        lambda x: x.get("disclosures", {}) if isinstance(x, dict) else {}
    )

    if "metrics" not in out.columns:
        out["metrics"] = None

    keep = [
        "event_id_norm",
        "condition",
        "persona",
        "engine",
        "summary_text",
        "drivers_used_norm",
        "driver_statements_norm",
        "action_recommendation_norm",
        "disclosures_norm",
        "metrics",
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep]


llm_norm_df = extract_llm_fields(llm_df)
tmpl_norm_df = extract_template_fields(tmpl_df) if len(tmpl_df) > 0 else pd.DataFrame()

combined_norm_df = pd.concat(
    [df for df in [llm_norm_df, tmpl_norm_df] if not df.empty],
    ignore_index=True,
)

print("LLM normalised rows:", len(llm_norm_df))
print("Template normalised rows:", len(tmpl_norm_df))
print("Combined rows:", len(combined_norm_df))

combined_norm_df.head()

LLM normalised rows: 200
Template normalised rows: 200
Combined rows: 400


,event_id_norm,condition,persona,engine,summary_text,drivers_used_norm,driver_statements_norm,action_recommendation_norm,disclosures_norm,metrics
0,3488961,llm,ops_triage,llm,Risk is LOW (score 0.005848). TransactionAmt i...,"[TransactionAmt, C5, C13, D1, D10]","[{'name': 'TransactionAmt', 'direction': '+', ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
1,3488968,llm,ops_triage,llm,Risk is LOW (score 0.000505). C5 reduces risk....,"[C5, card5, C9, C14, D1]","[{'name': 'C5', 'direction': '-', 'text': 'C5 ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
2,3488985,llm,ops_triage,llm,Risk is LOW (score 0.004906). C14 reduces risk...,"[C14, C5, C1, C9, card2]","[{'name': 'C14', 'direction': '-', 'text': 'C1...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
3,3489002,llm,ops_triage,llm,Risk is LOW (score 0.002055). D2 reduces risk....,"[D2, TransactionAmt, D1, C5, card2]","[{'name': 'D2', 'direction': '-', 'text': 'D2 ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
4,3489003,llm,ops_triage,llm,Risk is LOW (score 0.016481). card5 increases ...,"[card5, C5, D10, C13, TransactionDT]","[{'name': 'card5', 'direction': '+', 'text': '...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."


## Results: EO-grounded narrative governance evaluation

This section evaluates a reproducible run of the proposed EO → narrative → validator pipeline in the IEEE-CIS fraud setting. The analysis focuses on whether downstream narratives remain faithful to Evidence Objects (EOs), communicate uncertainty appropriately under limited evidence, and satisfy the closed-world governance constraints defined in the thesis.

The underlying fraud detector provides a credible substrate for explanation analysis rather than the primary object of optimisation. On the temporal split v1_temporal_q70_q85, the leakage-controlled LightGBM baseline achieved validation ROC-AUC of 0.8722 and PR-AUC of 0.4478, with test ROC-AUC of 0.8687 and PR-AUC of 0.4594. These results indicate that the explanation study is grounded in a non-trivial and reasonably competent fraud-scoring model, without claiming state-of-the-art predictive performance.

The full EO artifact contains 20,000 test-side records. The detailed narrative evaluation reported here is conducted on a 200-event sampled slice for the constrained-LLM condition and 200 corresponding template narratives for shared-event comparison. Within this sampled run, the constrained LLM narratives achieved perfect scores on the implemented first-order faithfulness checks: overlap@5 = 1.0, direction accuracy = 1.0, and action consistency = 1.0. Required evidence disclosures for thin-file cases were also present in all sampled narratives, yielding a thin-file disclosure rate of 1.0. Notebook-recomputed metrics matched the embedded metrics stored in the narrative artifacts, supporting the internal consistency of the evaluation pipeline.

These findings indicate that, under the current EO schema, constrained generation setup, and validator configuration, the EO-grounded narrative protocol behaves as intended on the sampled slice. The generated narratives preserved EO driver membership and sign, matched EO-recommended actions under the implemented evaluation logic, and included required evidence-limit disclosures under sparse-evidence conditions. In practical terms, this suggests that a tightly constrained explanation layer can deliver structurally faithful and governance-compliant narratives without introducing unsupported content.

The broader artifact-level evaluation strengthens this conclusion. Across the three sampled personas examined—ops triage, governance audit, and engineering debug—all constrained LLM outputs passed validation on the first attempt, with no retries and no recorded failure reasons. In the ops-triage top-5 grounding analysis, no unsupported driver leakage was detected, overlap with EO top drivers was perfect, and no sign errors were observed under the implemented checks. These results provide evidence that the closed-world grounding contract was preserved in the evaluated run and that the validator architecture was effective in enforcing the required narrative schema.

A compact perturbation experiment provides more direct evidence on narrative stability. A sampled subset of 100 EO rows was expanded into 300 perturbed variants covering three controlled perturbation types: action flips, thin-file disclosure toggles, and top-driver sign flips. The resulting narratives responded correctly to thin-file disclosure toggles in all evaluated cases and to top-driver sign flips in all evaluated cases where the target driver was mentioned, indicating that the explanation layer is responsive to relevant changes in the EO contract rather than merely replaying a static template. This strengthens the thesis support for stability by showing that the narrative output changes when governance-relevant EO fields are changed.

The action-perturbation results were more nuanced. Direct response to action flips was substantially weaker than response to disclosure and sign perturbations. However, qualitative inspection shows that this is largely explained by the thin-file guardrail policy encoded in the orchestration logic: when the perturbed EO still satisfied the thin-file or low-evidence condition, the narrative system frequently overrode the EO action field and emitted a step-up recommendation instead. This is an important governance result in its own right. It shows that explanation behaviour in this system is mediated not only by direct EO field values, but also by higher-priority policy rules applied to those values. Accordingly, lower action-flip response should not be interpreted simply as instability; rather, it reflects policy-conditioned responsiveness.

Cross-condition comparison against the deterministic template baseline showed exact agreement on the sampled shared events. On the 200 shared EO inputs, constrained LLM and template narratives matched exactly in action recommendation, driver set, and evidence-disclosure presence. This does not establish superiority of one condition over the other in readability or analyst utility, but it does show that the constrained LLM condition remained structurally aligned with the deterministic baseline while preserving the same EO-grounded content.

A related governance distinction emerges in the thin-file analysis. Narrative-level action consistency and disclosure compliance were both perfect in the sampled thin-file cases, yet the stricter thin-file action-validity metric remained at 0.0 under the policy rule encoded in the evaluation artifacts. This distinction is analytically important. It shows that explanation-level faithfulness and policy-level action validity are not the same thing: a narrative can be perfectly faithful to the EO and still surface an underlying governance issue in the EO’s recommended action under limited-evidence conditions. The present results therefore support strong claims about structural faithfulness and disclosure compliance, but only partial claims about the adequacy of thin-file action policy.

Taken together, the results support the thesis claim that governance-ready explanatory narratives can be produced when explanation generation is tightly coupled to an EO contract and protected by tiered validation. Within the tested configuration, the proposed pipeline achieved strong structural faithfulness, maintained closed-world grounding, and reliably communicated evidence limitations. The added perturbation analysis strengthens this claim by showing that the narrative layer is conditionally responsive to controlled EO changes, while also revealing that such responsiveness is shaped by explicit governance policy. The remaining limitations therefore concern not whether the protocol can enforce grounded explanation behaviour in a controlled setting, but how far those guarantees extend under broader perturbation regimes, richer evidence variation, and alternative policy configurations.

## Limitations

This notebook reports first-order governance evidence together with a compact perturbation experiment, rather than the full perturbation, randomisation, or counterfactual stress-testing programme proposed for the thesis. Evidence-strength variation within the attached EO–LLM sample remains limited, and the sampled LLM subset is still heavily concentrated in thin-file cases. In addition, embedded artifact metric names and notebook-recomputed metric names are not yet fully harmonised (for example, `overlap_at_K` versus `overlap_at_5`).

More broadly, the notebook evaluates explanation-governance behaviour under a reproducible offline run; it does not, by itself, establish production deployment readiness, portfolio-level calibration adequacy, or generalisation beyond the current experiment configuration.

The uniformly perfect validator and grounding metrics in this sampled run should be interpreted cautiously: they likely reflect a strongly constrained generation setting and a relatively narrow evaluation slice, rather than demonstrating unconstrained narrative robustness.

The perturbation experiment strengthens the stability analysis, but it remains compact in scope. It covers three controlled perturbation families on a 100-event sampled subset and therefore should be interpreted as evidence of conditional responsiveness under the implemented governance logic, not as an exhaustive robustness study.

In [13]:
# Cell 12 — build perturbed EO variants for a minimal stability experiment

from copy import deepcopy
import json
import random
from pathlib import Path
import pandas as pd

# ----------------------------
# Configuration
# ----------------------------
PERTURB_N = 100          # increase to 200 if you want
PERTURB_SEED = 42
WRITE_JSONL = True

# Optional: prefer rows with at least one top driver having an explicit +/- direction
REQUIRE_TOP1_DIRECTION = True

random.seed(PERTURB_SEED)

# ----------------------------
# Helpers
# ----------------------------
def _safe_top_drivers(x):
    return x if isinstance(x, list) else []

def _flip_action(action: str) -> str:
    # Conservative action perturbation map.
    # Adjust if your EO action space differs.
    action_map = {
        "allow": "review",
        "review": "block",
        "block": "review",
        "monitor": "review",
        "decline": "review",
    }
    return action_map.get(action, "review")

def _flip_direction(direction):
    if direction == "+":
        return "-"
    if direction == "-":
        return "+"
    return direction

def _row_has_top1_direction(row) -> bool:
    top_drivers = _safe_top_drivers(row.get("top_drivers"))
    if not top_drivers:
        return False
    top1 = top_drivers[0]
    return isinstance(top1, dict) and top1.get("direction") in {"+", "-"}

def _make_variant(base_row: dict, perturbation_type: str) -> dict:
    row = deepcopy(base_row)

    # Keep original ID around; create a variant-specific event_id for generation
    base_event_id = str(row["event_id"])
    row["source_event_id"] = base_event_id
    row["perturbation_type"] = perturbation_type
    row["is_perturbed"] = True

    if perturbation_type == "action_flip":
        row["recommended_action_class_original"] = row.get("recommended_action_class")
        row["recommended_action_class"] = _flip_action(row.get("recommended_action_class"))

    elif perturbation_type == "thin_file_toggle":
        row["thin_file_flag_original"] = row.get("thin_file_flag")
        row["thin_file_flag"] = not bool(row.get("thin_file_flag"))

        # Optional: nudge evidence_strength to remain semantically aligned
        # Only do this if your generator uses evidence_strength directly.
        if "evidence_strength" in row:
            if row["thin_file_flag"] is True:
                row["evidence_strength"] = "LOW"
            elif row["evidence_strength"] == "LOW":
                row["evidence_strength"] = "MEDIUM"

    elif perturbation_type == "top1_sign_flip":
        top_drivers = _safe_top_drivers(row.get("top_drivers"))
        if top_drivers and isinstance(top_drivers[0], dict):
            row["top_driver_name_original"] = top_drivers[0].get("name")
            row["top_driver_direction_original"] = top_drivers[0].get("direction")
            top_drivers[0]["direction"] = _flip_direction(top_drivers[0].get("direction"))
            row["top_driver_direction_perturbed"] = top_drivers[0].get("direction")
            row["top_drivers"] = top_drivers

    else:
        raise ValueError(f"Unknown perturbation_type: {perturbation_type}")

    row["event_id"] = f"{base_event_id}__{perturbation_type}"
    return row

# ----------------------------
# Sample source rows
# ----------------------------
candidate_df = eo_df.copy()

if REQUIRE_TOP1_DIRECTION:
    candidate_df = candidate_df[candidate_df.apply(_row_has_top1_direction, axis=1)].copy()

if len(candidate_df) < PERTURB_N:
    raise ValueError(
        f"Not enough candidate rows for perturbation sample. "
        f"Requested {PERTURB_N}, found {len(candidate_df)}."
    )

sampled_source_df = candidate_df.sample(n=PERTURB_N, random_state=PERTURB_SEED).copy()
sampled_source_df["source_event_id"] = sampled_source_df["event_id"].astype(str)

# ----------------------------
# Build perturbation variants
# ----------------------------
perturbation_types = ["action_flip", "thin_file_toggle", "top1_sign_flip"]

perturbed_rows = []
for _, row in sampled_source_df.iterrows():
    base_row = row.to_dict()
    for ptype in perturbation_types:
        perturbed_rows.append(_make_variant(base_row, ptype))

perturbed_eo_df = pd.DataFrame(perturbed_rows)

# ----------------------------
# Compact summary table
# ----------------------------
perturbation_build_summary_df = pd.DataFrame([
    {
        "n_source_rows": len(sampled_source_df),
        "n_perturbed_rows": len(perturbed_eo_df),
        "perturbations_per_row": len(perturbation_types),
        "perturbation_types": ", ".join(perturbation_types),
        "source_thin_file_rate": float(sampled_source_df["thin_file_flag"].mean()),
    }
])

display(perturbation_build_summary_df)

display(
    perturbed_eo_df[
        [
            "event_id",
            "source_event_id",
            "perturbation_type",
            "recommended_action_class",
            "thin_file_flag",
            "evidence_strength",
        ]
    ].head(12)
)

# ----------------------------
# Optional JSONL export
# ----------------------------
if WRITE_JSONL:
    perturb_dir = OUTPUT / "perturbation_eval"
    perturb_dir.mkdir(parents=True, exist_ok=True)

    perturb_jsonl_path = perturb_dir / f"eos_perturbed_n{PERTURB_N}.jsonl"

    with perturb_jsonl_path.open("w", encoding="utf-8") as f:
        for row in perturbed_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Wrote perturbed EO JSONL:", perturb_jsonl_path)

,n_source_rows,n_perturbed_rows,perturbations_per_row,perturbation_types,source_thin_file_rate
0,100,300,3,"action_flip, thin_file_toggle, top1_sign_flip",0.83


,event_id,source_event_id,perturbation_type,recommended_action_class,thin_file_flag,evidence_strength
0,3536688__action_flip,3536688,action_flip,review,True,LOW
1,3536688__thin_file_toggle,3536688,thin_file_toggle,allow,False,MEDIUM
2,3536688__top1_sign_flip,3536688,top1_sign_flip,allow,True,LOW
3,3497879__action_flip,3497879,action_flip,review,True,LOW
4,3497879__thin_file_toggle,3497879,thin_file_toggle,allow,False,MEDIUM
5,3497879__top1_sign_flip,3497879,top1_sign_flip,allow,True,LOW
6,3527865__action_flip,3527865,action_flip,review,True,LOW
7,3527865__thin_file_toggle,3527865,thin_file_toggle,allow,False,MEDIUM
8,3527865__top1_sign_flip,3527865,top1_sign_flip,allow,True,LOW
9,3494013__action_flip,3494013,action_flip,review,True,LOW


Wrote perturbed EO JSONL: /home/amurtagh/repos/miniature-fishstick/output/perturbation_eval/eos_perturbed_n100.jsonl


In [14]:
# Cell 13 — evaluate perturbed narrative outputs

from pathlib import Path
import pandas as pd
import json

# -------------------------------------------------
# Configuration: set this to the perturbed narrative output JSONL
# -------------------------------------------------

PERTURBED_LLM_PATH = OUTPUT / "perturbation_eval" / "narratives_ops_triage_llm.jsonl"

if not PERTURBED_LLM_PATH.exists():
    raise FileNotFoundError(
        f"Perturbed narrative file not found: {PERTURBED_LLM_PATH}\n"
        "Update PERTURBED_LLM_PATH to the actual generated perturbed narrative JSONL."
    )

# -------------------------------------------------
# Load perturbed narratives
# -------------------------------------------------
perturbed_llm_rows = read_jsonl(PERTURBED_LLM_PATH)
perturbed_llm_df = pd.DataFrame(perturbed_llm_rows)

required_cols = ["eo_event_id", "narrative"]
_assert_columns(perturbed_llm_df, required_cols, name="perturbed_llm_df")
perturbed_llm_df["eo_event_id"] = perturbed_llm_df["eo_event_id"].astype(str)

# -------------------------------------------------
# Merge perturbed EOs with perturbed narratives
# -------------------------------------------------
if "event_id" not in perturbed_eo_df.columns:
    raise AssertionError("perturbed_eo_df not found. Run the perturbation-construction cell first.")

perturbed_eo_df["event_id"] = perturbed_eo_df["event_id"].astype(str)

perturbed_eval_df = perturbed_eo_df.merge(
    perturbed_llm_df,
    left_on="event_id",
    right_on="eo_event_id",
    how="inner",
    suffixes=("_eo", "_llm"),
)

if perturbed_eval_df.empty:
    raise AssertionError(
        "Merged perturbed EO × perturbed LLM dataframe is empty. "
        "Check that the narrative generator used the perturbed EO event_id values."
    )

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def _narrative_action(narr):
    return narr.get("action_recommendation") if isinstance(narr, dict) else None

def _has_evidence_disclosure(narr):
    if not isinstance(narr, dict):
        return False
    disclosures = narr.get("disclosures", {})
    if not isinstance(disclosures, dict):
        return False
    val = disclosures.get("evidence")
    return isinstance(val, str) and len(val.strip()) > 0

def _driver_direction_lookup(narr):
    if not isinstance(narr, dict):
        return {}
    stmts = narr.get("driver_statements", [])
    if not isinstance(stmts, list):
        return {}
    return {
        d.get("name"): d.get("direction")
        for d in stmts
        if isinstance(d, dict) and d.get("name") is not None
    }

# -------------------------------------------------
# Derive response indicators
# -------------------------------------------------
perturbed_eval_df["narrative_action_recommendation"] = perturbed_eval_df["narrative"].apply(_narrative_action)
perturbed_eval_df["has_evidence_disclosure"] = perturbed_eval_df["narrative"].apply(_has_evidence_disclosure)
perturbed_eval_df["driver_direction_lookup"] = perturbed_eval_df["narrative"].apply(_driver_direction_lookup)

# Action-flip response
action_mask = perturbed_eval_df["perturbation_type"] == "action_flip"
perturbed_eval_df.loc[action_mask, "action_response_ok"] = (
    perturbed_eval_df.loc[action_mask, "narrative_action_recommendation"]
    == perturbed_eval_df.loc[action_mask, "recommended_action_class"]
)

# Thin-file toggle response
thin_mask = perturbed_eval_df["perturbation_type"] == "thin_file_toggle"
perturbed_eval_df.loc[thin_mask, "disclosure_toggle_response_ok"] = perturbed_eval_df.loc[thin_mask].apply(
    lambda r: (
        r["has_evidence_disclosure"] is True if bool(r["thin_file_flag"]) else r["has_evidence_disclosure"] is False
    ),
    axis=1,
)

# Top1 sign-flip response
sign_mask = perturbed_eval_df["perturbation_type"] == "top1_sign_flip"
perturbed_eval_df.loc[sign_mask, "top1_sign_response_ok"] = perturbed_eval_df.loc[sign_mask].apply(
    lambda r: (
        r["driver_direction_lookup"].get(r.get("top_driver_name_original"))
        == r.get("top_driver_direction_perturbed")
    ),
    axis=1,
)

perturbed_eval_df.loc[sign_mask, "top1_driver_mentioned"] = perturbed_eval_df.loc[sign_mask].apply(
    lambda r: r.get("top_driver_name_original") in r["driver_direction_lookup"],
    axis=1,
)

# -------------------------------------------------
# Summary table
# -------------------------------------------------
perturbation_response_summary_df = pd.DataFrame([
    {
        "perturbation_type": "action_flip",
        "n": int(action_mask.sum()),
        "expected_response_rate": float(perturbed_eval_df.loc[action_mask, "action_response_ok"].mean()) if action_mask.sum() else None,
        "failure_rate": 1.0 - float(perturbed_eval_df.loc[action_mask, "action_response_ok"].mean()) if action_mask.sum() else None,
    },
    {
        "perturbation_type": "thin_file_toggle",
        "n": int(thin_mask.sum()),
        "expected_response_rate": float(perturbed_eval_df.loc[thin_mask, "disclosure_toggle_response_ok"].mean()) if thin_mask.sum() else None,
        "failure_rate": 1.0 - float(perturbed_eval_df.loc[thin_mask, "disclosure_toggle_response_ok"].mean()) if thin_mask.sum() else None,
    },
    {
        "perturbation_type": "top1_sign_flip",
        "n": int(sign_mask.sum()),
        "expected_response_rate": float(perturbed_eval_df.loc[sign_mask, "top1_sign_response_ok"].mean()) if sign_mask.sum() else None,
        "failure_rate": 1.0 - float(perturbed_eval_df.loc[sign_mask, "top1_sign_response_ok"].mean()) if sign_mask.sum() else None,
        "top1_driver_mention_rate": float(perturbed_eval_df.loc[sign_mask, "top1_driver_mentioned"].mean()) if sign_mask.sum() else None,
    },
])

display(perturbation_response_summary_df)

# -------------------------------------------------
# Small qualitative sample for inspection
# -------------------------------------------------
qual_cols = [
    c for c in [
        "event_id",
        "source_event_id",
        "perturbation_type",
        "recommended_action_class",
        "thin_file_flag",
        "top_driver_name_original",
        "top_driver_direction_original",
        "top_driver_direction_perturbed",
        "narrative_action_recommendation",
        "has_evidence_disclosure",
        "narrative",
    ]
    if c in perturbed_eval_df.columns
]

display(perturbed_eval_df[qual_cols].head(12))

print("Perturbed EO rows:", len(perturbed_eo_df))
print("Perturbed narrative rows:", len(perturbed_llm_df))
print("Merged perturbed eval rows:", len(perturbed_eval_df))

,perturbation_type,n,expected_response_rate,failure_rate,top1_driver_mention_rate
0,action_flip,100,0.17,0.83,NaN
1,thin_file_toggle,100,1.00,0.00,NaN
2,top1_sign_flip,100,1.00,0.00,1.0


,event_id,source_event_id,perturbation_type,recommended_action_class,thin_file_flag,top_driver_name_original,top_driver_direction_original,top_driver_direction_perturbed,narrative_action_recommendation,has_evidence_disclosure,narrative
0,3536688__action_flip,3536688,action_flip,review,True,NaN,NaN,NaN,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
1,3536688__thin_file_toggle,3536688,thin_file_toggle,allow,False,NaN,NaN,NaN,allow,False,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
2,3536688__top1_sign_flip,3536688,top1_sign_flip,allow,True,TransactionAmt,+,-,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
3,3497879__action_flip,3497879,action_flip,review,True,NaN,NaN,NaN,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
4,3497879__thin_file_toggle,3497879,thin_file_toggle,allow,False,NaN,NaN,NaN,allow,False,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
5,3497879__top1_sign_flip,3497879,top1_sign_flip,allow,True,TransactionAmt,+,-,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
6,3527865__action_flip,3527865,action_flip,review,True,NaN,NaN,NaN,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
7,3527865__thin_file_toggle,3527865,thin_file_toggle,allow,False,NaN,NaN,NaN,allow,False,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
8,3527865__top1_sign_flip,3527865,top1_sign_flip,allow,True,D2,-,+,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."
9,3494013__action_flip,3494013,action_flip,review,True,NaN,NaN,NaN,step-up,True,"{'persona': 'ops_triage', 'summary': 'Risk is ..."


Perturbed EO rows: 300
Perturbed narrative rows: 300
Merged perturbed eval rows: 300


## Perturbation-response interpretation

The perturbation experiment provides direct evidence on narrative responsiveness under controlled EO changes. Two perturbation families behaved as expected across the sampled slice: when the thin-file flag was toggled, the presence of evidence disclosures changed accordingly in all cases; and when the sign of the top-ranked EO driver was flipped, the corresponding narrative driver-direction statement also changed in all cases where the driver was mentioned. These results strengthen support for RQ2 by showing that the narrative layer is responsive to relevant changes in the EO contract rather than merely reproducing a static template.

Action perturbations behaved differently. The action-flip response rate was low because many perturbed rows still satisfied the thin-file / low-evidence condition, under which the guardrail policy encoded in the orchestration logic overrides the EO action and forces a `step-up` recommendation. Accordingly, the low action-flip response rate should not be interpreted simply as narrative instability; rather, it reflects an interaction between EO perturbation and the higher-priority thin-file action policy. This is substantively useful for the thesis because it shows that narrative responsiveness is conditional not only on EO field changes, but also on the governance rules applied to those fields.

In [15]:
# Cell 14 — Thesis-ready RQ / Hypothesis coverage table

rq_hypothesis_coverage_df = pd.DataFrame([
    {
        "Question / Hypothesis": "RQ1",
        "Topic": "Faithfulness",
        "What the proposal set out to test": (
            "Whether narratives preserve EO driver membership, direction, and recommended action class."
        ),
        "Evidence provided in this notebook": (
            "Directly evaluated using overlap@5, direction accuracy, and action consistency "
            "on the EO × LLM merged sample."
        ),
        "Assessment in this submission": "Addressed",
        "Comment for thesis write-up": (
            "This remains the strongest-supported research question in the current submission."
        ),
    },
    {
        "Question / Hypothesis": "RQ2",
        "Topic": "Stability",
        "What the proposal set out to test": (
            "Whether narratives remain stable under perturbations and degrade appropriately under sanity checks."
        ),
        "Evidence provided in this notebook": (
            "Direct perturbation evidence from a 100-event sampled EO subset expanded into 300 perturbed variants, "
            "plus thin-file and evidence-strength subgroup summaries. The notebook evaluates response to action flips, "
            "thin-file disclosure toggles, and top-driver sign flips."
        ),
        "Assessment in this submission": "Addressed",
        "Comment for thesis write-up": (
            "This is a compact rather than exhaustive stability study, but it provides direct perturbation-based evidence."
        ),
    },
    {
        "Question / Hypothesis": "RQ3",
        "Topic": "Decision utility",
        "What the proposal set out to test": (
            "Whether narratives provide better operational understanding than raw driver lists "
            "(for example via simulatability or forward-prediction tasks)."
        ),
        "Evidence provided in this notebook": (
            "Indirect proxy only: narratives contain summary, drivers, disclosures, and action fields, "
            "but no formal utility task is implemented."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Present this as proxy evidence rather than a full decision-utility evaluation."
        ),
    },
    {
        "Question / Hypothesis": "RQ4",
        "Topic": "Drift awareness",
        "What the proposal set out to test": (
            "Whether drift and monitoring disclosures reduce over-trust and improve action calibration."
        ),
        "Evidence provided in this notebook": (
            "Not directly analysed in the current notebook."
        ),
        "Assessment in this submission": "Not addressed",
        "Comment for thesis write-up": (
            "Treat as future work or note that drift-focused analysis is outside the present run."
        ),
    },
    {
        "Question / Hypothesis": "RQ5",
        "Topic": "Thin-file robustness",
        "What the proposal set out to test": (
            "Whether narratives behave appropriately under sparse evidence, especially through disclosure "
            "and calibrated action recommendations."
        ),
        "Evidence provided in this notebook": (
            "Direct thin-file disclosure compliance checks, thin-file and evidence-strength stratified summaries, "
            "and perturbation-based disclosure toggle evidence."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Disclosure behaviour is well supported; action recalibration under sparse evidence is evidenced more strongly "
            "than before, but remains intertwined with policy overrides."
        ),
    },
    {
        "Question / Hypothesis": "RQ6",
        "Topic": "Portability",
        "What the proposal set out to test": (
            "Whether the EO → narrative protocol generalises to optional cyber or crypto slices."
        ),
        "Evidence provided in this notebook": (
            "No portability experiment is included in the current notebook."
        ),
        "Assessment in this submission": "Not addressed",
        "Comment for thesis write-up": (
            "This was an optional proposal item and can be formally marked as out of scope."
        ),
    },
    {
        "Question / Hypothesis": "H1",
        "Topic": "LLM readability advantage",
        "What the proposal set out to test": (
            "Whether constrained LLM narratives are more readable than templates without loss of faithfulness."
        ),
        "Evidence provided in this notebook": (
            "Faithfulness comparison infrastructure exists, but readability is not formally measured."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Support this only qualitatively unless a readability proxy or annotation is added."
        ),
    },
    {
        "Question / Hypothesis": "H2",
        "Topic": "Value of constraints",
        "What the proposal set out to test": (
            "Whether removing grounding and validator constraints reduces faithfulness and increases invalid claims."
        ),
        "Evidence provided in this notebook": (
            "No unconstrained ablation is included, but the perturbation experiment provides indirect evidence "
            "that constrained narratives remain responsive to EO changes under the implemented guardrail policy."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "This does not substitute for a true unconstrained ablation, so avoid overstating the result."
        ),
    },
    {
        "Question / Hypothesis": "H3",
        "Topic": "Effect of drift / low-evidence messaging",
        "What the proposal set out to test": (
            "Whether uncertainty and drift messaging reduces over-confident or overly aggressive actions."
        ),
        "Evidence provided in this notebook": (
            "Disclosure presence is checked directly, and the perturbation analysis shows that disclosure behaviour "
            "tracks thin-file status changes. Behavioural consequences are not directly evaluated."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Current evidence supports disclosure responsiveness more than downstream behavioural effect."
        ),
    },
    {
        "Question / Hypothesis": "H4",
        "Topic": "Thin-file action calibration",
        "What the proposal set out to test": (
            "Whether limited-evidence messaging reduces inappropriate hard actions while preserving alignment."
        ),
        "Evidence provided in this notebook": (
            "Thin-file disclosures are present where required, and perturbation results show that action behaviour "
            "is strongly shaped by the thin-file guardrail policy under low-evidence conditions."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "This is now better supported, but the current evidence still reflects one implemented guardrail policy "
            "rather than a broader comparative calibration study."
        ),
    },
])

rq_hypothesis_coverage_df

,Question / Hypothesis,Topic,What the proposal set out to test,Evidence provided in this notebook,Assessment in this submission,Comment for thesis write-up
0,RQ1,Faithfulness,Whether narratives preserve EO driver membersh...,"Directly evaluated using overlap@5, direction ...",Addressed,This remains the strongest-supported research ...
1,RQ2,Stability,Whether narratives remain stable under perturb...,Direct perturbation evidence from a 100-event ...,Addressed,This is a compact rather than exhaustive stabi...
2,RQ3,Decision utility,Whether narratives provide better operational ...,Indirect proxy only: narratives contain summar...,Partially addressed,Present this as proxy evidence rather than a f...
3,RQ4,Drift awareness,Whether drift and monitoring disclosures reduc...,Not directly analysed in the current notebook.,Not addressed,Treat as future work or note that drift-focuse...
4,RQ5,Thin-file robustness,Whether narratives behave appropriately under ...,"Direct thin-file disclosure compliance checks,...",Partially addressed,Disclosure behaviour is well supported; action...
5,RQ6,Portability,Whether the EO → narrative protocol generalise...,No portability experiment is included in the c...,Not addressed,This was an optional proposal item and can be ...
6,H1,LLM readability advantage,Whether constrained LLM narratives are more re...,"Faithfulness comparison infrastructure exists,...",Partially addressed,Support this only qualitatively unless a reada...
7,H2,Value of constraints,Whether removing grounding and validator const...,"No unconstrained ablation is included, but the...",Partially addressed,This does not substitute for a true unconstrai...
8,H3,Effect of drift / low-evidence messaging,Whether uncertainty and drift messaging reduce...,"Disclosure presence is checked directly, and t...",Partially addressed,Current evidence supports disclosure responsiv...
9,H4,Thin-file action calibration,Whether limited-evidence messaging reduces ina...,Thin-file disclosures are present where requir...,Partially addressed,"This is now better supported, but the current ..."


In [16]:
# Cell 15 — validator outcomes across personas

validator_summary_df = pd.DataFrame([
    {
        "persona": "ops_triage",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
    {
        "persona": "governance_audit",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
    {
        "persona": "engineering_debug",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
])

validator_summary_df

,persona,n,validator_pass_count,validator_pass_rate,zero_retry_count,zero_retry_rate,failure_reason_count
0,ops_triage,200,200,1.0,200,1.0,0
1,governance_audit,200,200,1.0,200,1.0,0
2,engineering_debug,200,200,1.0,200,1.0,0


In [17]:
# Cell 16 — Artifact-level grounding and faithfulness summary (ops_triage top-5)

artifact_grounding_summary_df = pd.DataFrame([
    {
        "slice": "overall",
        "n": 200,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": 0.0,
    },
    {
        "slice": "thin",
        "n": 182,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": None,
    },
    {
        "slice": "thick",
        "n": 18,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": None,
    },
])

artifact_grounding_summary_df

,slice,n,overlap_at_5_mean,mention_any_top5_rate,any_sign_error_rate,leak_any_rate
0,overall,200,1.0,1.0,0.0,0.0
1,thin,182,1.0,1.0,0.0,NaN
2,thick,18,1.0,1.0,0.0,NaN


In [18]:
# Cell 17 — Thin-file action governance summary (based on sampled eval artifacts)

thinfile_action_governance_df = pd.DataFrame([
    {
        "metric": "action_consistency_rate",
        "value": 1.0,
        "interpretation": "Narrative action matches EO recommended action."
    },
    {
        "metric": "evidence_disclosure_rate",
        "value": 1.0,
        "interpretation": "Thin-file evidence disclosure is present when required in sampled outputs."
    },
    {
        "metric": "thin_file_action_valid_rate",
        "value": 0.0,
        "interpretation": (
            "Under the stricter thin-file action-validity rule encoded in the artifact metrics, "
            "sampled outputs do not satisfy the thin-file action policy."
        )
    },
])

thinfile_action_governance_df

,metric,value,interpretation
0,action_consistency_rate,1.0,Narrative action matches EO recommended action.
1,evidence_disclosure_rate,1.0,Thin-file evidence disclosure is present when ...
2,thin_file_action_valid_rate,0.0,Under the stricter thin-file action-validity r...


## Interpreting thin-file governance results

The thin-file results evaluate two different things. `action_consistency_rate` measures whether the narrative recommendation matches the EO’s recommended action, whereas `thin_file_action_valid_rate` reflects a stricter governance policy about whether that action is itself acceptable under limited-evidence conditions. Accordingly, a narrative can be perfectly faithful to the EO while still surfacing a policy-level issue in thin-file action design.

## Additional artifact-level governance findings

The broader narrative-evaluation artifacts extend the main EO×LLM notebook analysis beyond the ops-triage merged sample in three ways. First, constrained LLM outputs passed the validator on the first attempt in all sampled cases across the three personas examined, with no retries and no recorded failure reasons. Second, closed-world grounding remained intact in the sampled ops-triage top-5 evaluation: no unsupported driver leakage was detected, overlap with EO top drivers was perfect, and no sign errors were detected under the current sign-check procedure. Third, the thin-file analysis reveals an important governance distinction: disclosure presence and EO-consistent action recommendation can both be perfect while a stricter thin-file action-validity criterion still fails. This suggests that structural faithfulness and policy-valid action calibration should be treated as related but distinct evaluation targets.

In [19]:
# Cell 18 — baseline model performance summary

with METRICS_PATH.open("r", encoding="utf-8") as f:
    baseline_metrics_raw = json.load(f)

baseline_run_summary_df = pd.DataFrame([{
    "model": baseline_metrics_raw["model"],
    "split_id": baseline_metrics_raw["split_id"],
    "best_iteration": baseline_metrics_raw["best_iteration"],
    "n_train": baseline_metrics_raw["n_rows"]["train"],
    "n_val": baseline_metrics_raw["n_rows"]["val"],
    "n_test": baseline_metrics_raw["n_rows"]["test"],
    "n_features": baseline_metrics_raw["n_features"],
}])

baseline_metrics_df = pd.DataFrame([
    {
        "split": "validation",
        "roc_auc": baseline_metrics_raw["val"]["roc_auc"],
        "pr_auc": baseline_metrics_raw["val"]["pr_auc"],
    },
    {
        "split": "test",
        "roc_auc": baseline_metrics_raw["test"]["roc_auc"],
        "pr_auc": baseline_metrics_raw["test"]["pr_auc"],
    },
])

baseline_run_summary_df, baseline_metrics_df

(                       model             split_id  best_iteration  n_train  \
 0  lgbm_numeric_v1_subsample  v1_temporal_q70_q85             539    50000   
 
    n_val  n_test  n_features  
 0  20000   20000          40  ,
         split   roc_auc    pr_auc
 0  validation  0.872179  0.447806
 1        test  0.868670  0.459436)

## Baseline model context

The explanation-governance analysis is grounded in a leakage-controlled LightGBM baseline trained on the IEEE-CIS temporal split `v1_temporal_q70_q85`. Test performance is sufficient for the thesis purpose (ROC-AUC 0.8687; PR-AUC 0.4594), supporting use of the detector as a credible substrate for evaluating EO-grounded narratives and validator behaviour.

The purpose of reporting these metrics is not to claim state-of-the-art fraud detection performance, but to show that the explanation analysis is attached to a non-trivial and reasonably competent fraud-scoring model.

In [20]:
# Cell 19 — template vs constrained LLM comparison on shared events

def _norm_list(x):
    return x if isinstance(x, list) else []

def _has_evidence_text(d):
    if not isinstance(d, dict):
        return False
    val = d.get("evidence")
    return isinstance(val, str) and len(val.strip()) > 0

if not compare_df.empty:
    compare_condition_summary_df = pd.DataFrame([{
        "n_shared_events": len(compare_df),
        "action_match_rate_llm_vs_template": (
            compare_df["narrative_action_recommendation_llm"]
            == compare_df["narrative_action_recommendation_tmpl"]
        ).mean(),
        "drivers_used_exact_match_rate": compare_df.apply(
            lambda r: _norm_list(r["narrative_drivers_used_llm"]) == _norm_list(r["narrative_drivers_used_tmpl"]),
            axis=1
        ).mean(),
        "evidence_disclosure_match_rate": compare_df.apply(
            lambda r: _has_evidence_text(r["narrative_disclosures_llm"]) == _has_evidence_text(r["narrative_disclosures_tmpl"]),
            axis=1
        ).mean(),
    }])
    display(compare_condition_summary_df)
else:
    print("compare_df is empty; no shared-event condition comparison available.")

,n_shared_events,action_match_rate_llm_vs_template,drivers_used_exact_match_rate,evidence_disclosure_match_rate
0,200,1.0,1.0,1.0


In [21]:
# Cell 20 — export thesis tables

EXPORT_DIR = OUTPUT / "thesis_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

llm_summary_df.to_csv(EXPORT_DIR / "llm_summary.csv", index=False)
rq_hypothesis_coverage_df.to_csv(EXPORT_DIR / "rq_hypothesis_coverage.csv", index=False)
artifact_grounding_summary_df.to_csv(EXPORT_DIR / "artifact_grounding_summary.csv", index=False)
thinfile_action_governance_df.to_csv(EXPORT_DIR / "thinfile_action_governance.csv", index=False)
baseline_run_summary_df.to_csv(EXPORT_DIR / "baseline_run_summary.csv", index=False)
baseline_metrics_df.to_csv(EXPORT_DIR / "baseline_metrics.csv", index=False)
validator_summary_df.to_csv(EXPORT_DIR / "validator_summary.csv", index=False)
perturbation_response_summary_df.to_csv(EXPORT_DIR / "perturbation_response_summary.csv", index=False)
perturbation_build_summary_df.to_csv(EXPORT_DIR / "perturbation_build_summary.csv", index=False)

if 'compare_condition_summary_df' in globals():
    compare_condition_summary_df.to_csv(EXPORT_DIR / "compare_condition_summary.csv", index=False)

if not compare_df.empty:
    compare_df.head(20).to_csv(EXPORT_DIR / "llm_vs_template_comparison_detail.csv", index=False)

print("Exported thesis tables to:", EXPORT_DIR)
for p in sorted(EXPORT_DIR.glob("*.csv")):
    print("-", p.name)

Exported thesis tables to: /home/amurtagh/repos/miniature-fishstick/output/thesis_tables
- artifact_grounding_summary.csv
- baseline_metrics.csv
- baseline_run_summary.csv
- compare_condition_summary.csv
- llm_summary.csv
- llm_vs_template_comparison_detail.csv
- perturbation_build_summary.csv
- perturbation_response_summary.csv
- rq_hypothesis_coverage.csv
- thinfile_action_governance.csv
- validator_summary.csv


## Cross-condition comparison note

This comparison does not attempt to measure readability or human preference. Instead, it tests whether constrained LLM narratives remain structurally aligned with the deterministic template baseline on shared EO inputs.